In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pickle 
import seaborn as sns
import numpy as np

In [ ]:
df_path = "/project3/data/results/eggnog/results/eggnog_output.emapper.annotations"

In [ ]:
emapperannot = pd.read_csv(df_path, sep="\t", skiprows=4)

In [ ]:
pcompPath = "/project3/data/results/eggnog/results/keggCompletness/methane_pathways.tsv"

In [ ]:
dfComp = pd.read_csv(pcompPath, sep="\t")

In [ ]:
### PLOTTING 

# Filter rows with "Energy metabolism" in 'pathway_class'
df_energy = dfComp[dfComp['pathway_class'].str.contains("Energy metabolism", na=False)].copy()

# Create a new column 'pathwayClean' as the last element from 'pathway_class'
df_energy['pathwayClean'] = df_energy['pathway_class'].str.split(';').str[-1].str.strip()

# Ensure 'completeness' is numeric
df_energy['completeness'] = pd.to_numeric(df_energy['completeness'], errors='coerce')

# Create 'pathwayShort' by taking the first part of 'pathway_name'
df_energy['pathwayShort'] = df_energy['pathway_name'].str.split(',', n=1).str[0]

# Compute a global maximum for completeness to enforce a consistent x-axis scale
global_max = df_energy['completeness'].max()

# Get unique pathwayClean values
unique_pathways = df_energy['pathwayClean'].unique()

final_df = pd.DataFrame()

# Loop over each unique pathwayClean to sort and drop duplicates while preserving the order
for pathway in unique_pathways:
    sub_df = df_energy[df_energy['pathwayClean'] == pathway].copy()
    sub_df = sub_df.sort_values(by='completeness', ascending=False)
    sub_df.drop_duplicates(subset=["pathwayShort"], inplace=True)
    final_df = pd.concat([final_df, sub_df], ignore_index=True)

# Create a color palette mapping for each pathwayClean using seaborn
palette = sns.color_palette("pastel", n_colors=len(unique_pathways))
pathway_color_map = dict(zip(unique_pathways, palette))

# Map each row in final_df to its corresponding color
bar_colors = final_df['pathwayClean'].map(pathway_color_map)

# Create a figure
fig, ax = plt.subplots(figsize=(12, 5))

# Generate positions for the bars
y_positions = list(range(len(final_df)))

# Plot horizontal bar chart with custom colors
ax.barh(y_positions, final_df['completeness'], height=0.8, align='center', color=bar_colors)

# Set a common x-axis scale
ax.set_xlim(0, global_max)

# Set y-ticks to correspond to pathwayShort labels
ax.set_yticks(y_positions)
ax.set_yticklabels(final_df['pathwayShort'], fontsize=12)

# Add labels and title
ax.set_xlabel('Completeness (%)', fontsize=12)
ax.set_title('', fontsize=12)

# Add a legend for the hue mapping
handles = [plt.Rectangle((0, 0), 1, 1, color=pathway_color_map[p]) for p in unique_pathways]
ax.legend(handles, unique_pathways, title="Energy Metabolism", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig("/home/projects2/metagnm_asm/project3/scripts/plotting-scripts/plots/eggnogAnnotCompleteness.svg", format="svg", bbox_inches="tight")
plt.show()


In [ ]:
with open('mgen_dfs.pkl', 'rb') as f:
    mgen = pickle.load(f)

In [ ]:
# Ensure no missing values in emapperannot
emapperannot.dropna(inplace=True)

results_list = []

# Loop over each unique pathwayClean value in final_df
for pathway in final_df['pathwayClean'].unique():
    # Filter rows for the specific pathwayClean
    df_subset = final_df[final_df['pathwayClean'] == pathway]
    
    # Gather all matching_ko codes into a list, ignoring any NaN values
    ko_list = []
    for ko_entry in df_subset['matching_ko'].dropna():
        ko_list.extend([ko.strip() for ko in ko_entry.split(',') if ko.strip() != ''])
    
    # Convert to a set for uniqueness and sort
    unique_kos = sorted(set(ko_list))
    
    # Filter emapperannot where KEGG_ko column contains any of the unique_kos
    emapperannot_q_list = emapperannot[
        emapperannot["KEGG_ko"].apply(lambda x: any(item in x for item in unique_kos))
    ]["#query"].values.tolist()
    
    # Process the query IDs (e.g., remove prefix up to the first underscore)
    protein_ids = [x.split("_", 1)[1] for x in emapperannot_q_list]
    
    # Filter mgen (or mgen_dfs) for rows where the "query" column matches any processed identifier
    matches = mgen[mgen["query"].isin(protein_ids)]
    
    # Calculate medians for each metric if matches exist
    median_fident = np.median(matches["fident"]) if not matches.empty else None
    
    min_fident = np.min(matches["fident"]) if not matches.empty else None
    
    median_qplddt = np.median(matches["qplddt"]) if not matches.empty else None
    median_tplddt = np.median(matches["tplddt"]) if not matches.empty else None
    median_qtmscore = np.median(matches["qtmscore"]) if not matches.empty else None
    
    # Count the number of unique matched queries
    unique_matches = matches["query"].nunique()
    
    # Create a dictionary holding the pathway and its metrics
    results = {
        "Pathway": pathway,
        "Median similarity": round(median_fident, 2) if median_fident is not None else None,
        "Median assembly pLDDT": round(median_qplddt, 2) if median_qplddt is not None else None,
        "Median database pLDDT": round(median_tplddt, 2) if median_tplddt is not None else None,
        "Median TM score": round(median_qtmscore, 2) if median_qtmscore is not None else None,
        "Total unique matches": unique_matches
    }
    
    results_list.append(results)

# Create a final DataFrame with all results
final_results_df = pd.DataFrame(results_list)

# Convert the final DataFrame to a LaTeX table
final_latex_table = final_results_df.to_latex(index=False)
print(final_latex_table)
